# Delft3D Physics-Informed Graph Neural Network

This notebook is configured to run on Kaggle. It imports the model from `pignn_model.py` and runs the time-stepping simulation using true boundary conditions.

In [ ]:
# 1. Install Dependencies
!pip install torch-geometric netCDF4 xarray scipy matplotlib

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 2. Import the PIGNN model
from pignn_model import extract_graph_from_netcdf, load_friction_xyz, load_boundary_pli, load_boundary_bc, RiverPIGNN, physics_loss_swe

### Parse True Data

In [ ]:
# PATHS (adjust if needed)
netcdf_path = "../data/input/FlowFM_net.nc"
friction_path = "../data/input/frictioncoefficient.xyz"
pli_port = "../data/input/port block.pli"
pli_garonne = "../data/input/garonne.pli"
pli_dordogne = "../data/input/dordogne.pli"
bc_port = "../data/input/WaterLevel.bc"
bc_garonne = "../data/input/garonne.bc"
bc_dordogne = "../data/input/dordogne.bc"

try:
    node_coords, edge_index, edge_attr, node_z = extract_graph_from_netcdf(netcdf_path)
    num_nodes = node_coords.size(0)
    print(f"Graph successfully extracted! Nodes: {num_nodes}")

    # Load real friction map
    friction = load_friction_xyz(friction_path, node_coords)

    # Load Boundary Nodes
    bnd_port = load_boundary_pli(pli_port, node_coords)
    bnd_garonne = load_boundary_pli(pli_garonne, node_coords)
    bnd_dordogne = load_boundary_pli(pli_dordogne, node_coords)

    # Load Time Series
    t_port, v_port = load_boundary_bc(bc_port)
    t_garonne, v_garonne = load_boundary_bc(bc_garonne)
    t_dordogne, v_dordogne = load_boundary_bc(bc_dordogne)

except FileNotFoundError as e:
    print(f"File not found error: {e}. Please check the paths.")
    num_nodes = 0

In [ ]:
if num_nodes > 0:
    node_coords = node_coords.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    node_z = node_z.to(device)
    friction = friction.to(device)

    model = RiverPIGNN(node_features_dim=5, hidden_dim=64, edge_dim=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # Simulation Parameters
    dt = 3600.0  # 1 hour
    max_time = t_port[-1] if len(t_port) > 0 else 36000
    current_time = 0.0

    # Initial State Setup 
    initial_water_level = float(torch.max(node_z)) + 2.0
    state_t = torch.zeros((num_nodes, 5), dtype=torch.float32).to(device)
    state_t[:, 0] = initial_water_level  # eta
    state_t[:, 3] = node_z               # bathymetry
    state_t[:, 4] = friction             # friction

    def get_interp_val(t, times, values):
        return np.interp(t, times, values)

    print("Starting Dynamic Physics Solver...")
    
    while current_time < max_time:
        model.train()
        optimizer.zero_grad()
        
        # 1. Inject True Boundary Conditions at time t
        state_t[bnd_port, 0] = get_interp_val(current_time, t_port, v_port)
        state_t[bnd_garonne, 1] = get_interp_val(current_time, t_garonne, v_garonne) * 0.001
        state_t[bnd_dordogne, 1] = get_interp_val(current_time, t_dordogne, v_dordogne) * 0.001

        # 2. Predict next state t+dt
        predicted_state_next = model(state_t, edge_index, edge_attr)
        
        # 3. Compute Physics Loss
        p_loss = physics_loss_swe(state_t, predicted_state_next, edge_index, edge_attr, dt=dt)
        p_loss.backward()
        optimizer.step()
        
        if current_time % (3600*4) == 0: 
            print(f"Time: {current_time/3600:.1f}h - Physics Loss: {p_loss.item():.6f}")
            
        # Step time forward
        state_t = predicted_state_next.detach().clone()
        state_t[:, 3] = node_z
        state_t[:, 4] = friction
        current_time += dt

    print("Simulation Complete!")
    torch.save(model.state_dict(), "pignn_weights_sim.pth")